In [4]:
# Importaciones necesarias
import pandas as pd
import numpy as np
import warnings
from itertools import combinations
import json
# Suprimir warning específico
warnings.filterwarnings('ignore',
                       message='Environment variable "XPC_SERVICE_NAME" redefined by R',
                       category=UserWarning)

import rpy2.robjects as robjects
from rpy2.robjects.conversion import get_conversion, localconverter
from rpy2.robjects.pandas2ri import converter
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# Activar conversión automática entre pandas y R
# pandas2ri.activate()


In [5]:
def perform_differential_expression(log_data_filtered, metadata, clinical_attribute):
    """
    Perform Differential Expression Analysis using limma (R package) via rpy2.
    """

    # Import R packages
    limma = importr('limma')  # limma for differential expression
    stats = importr('stats')  # stats for model matrix
    base = importr('base')    # base R functions

    # 1. Validation: Ensure the clinical attribute has at least two categories
    # This is necessary because differential expression requires at least two groups to compare.
    unique_values = metadata[clinical_attribute].dropna().unique()
    if len(unique_values) < 2:
        raise ValueError(f"At least two categories are required in '{clinical_attribute}'.")

    # 2. Group vector creation
    # Converts the clinical attribute to a categorical variable (factor in R).
    # This tells the model to treat the values as groups, not as numeric values.
    group_values = metadata[clinical_attribute].astype(str).values
    group_levels = sorted(np.unique(group_values))  # Get all unique group names sorted
    group_dict = {k: i for i, k in enumerate(group_levels)}  # Map group names to indices (not strictly needed)
    group_factor = pd.Categorical(group_values, categories=group_levels)

    # 3. Conversion to R objects
    # Converts the filtered log-expression data (Pandas DataFrame) to an R matrix.
    with localconverter(get_conversion() + converter):
        r_log_data = pandas2ri.py2rpy(log_data_filtered)
    # Convert the group factor to an R factor vector
    r_group = robjects.FactorVector(group_factor)

    # 4. Design matrix construction (no intercept)
    # The design matrix encodes the group structure for the linear model.
    # Using '~ 0 + group' means no intercept: each group gets its own column.
    formula = robjects.Formula('~ 0 + group')
    env = robjects.Environment()
    env['group'] = r_group
    r_design = stats.model_matrix(formula, env)
    r_design.colnames = robjects.StrVector(group_levels)  # Set column names to group names

    # 5. Linear model fitting with limma
    # Fit the linear model to estimate mean expression for each gene in each group.
    fit = limma.lmFit(r_log_data, r_design)

    # 6. Create all possible pairwise contrasts (all-vs-all)
    # For each pair of groups, create a contrast expression like 'groupB - groupA'.
    pares = list(combinations(group_levels, 2))
    contrastes = [f"{b} - {a}" for a, b in pares]
    contrast_matrix = limma.makeContrasts(
        contrasts=robjects.StrVector(contrastes),
        levels=r_design
    )

    # 7. Apply contrasts and empirical Bayes moderation
    # Apply the contrasts to the fitted model, then use eBayes to stabilize variance estimates.
    fit2 = limma.contrasts_fit(fit, contrast_matrix)
    fit2 = limma.eBayes(fit2)

    # 8. Extract results for the first contrast
    # Get the table of differential expression results for the first contrast (logFC, p-value, adjusted p-value, etc.).
    results = limma.topTable(
        fit2,
        coef=1,  # First contrast
        number=robjects.r('Inf'),  # All genes
        adjust_method="BH"  # Benjamini-Hochberg adjustment
    )
    # Convert the R data frame to a Pandas DataFrame for further analysis in Python.
    results_df = pandas2ri.rpy2py(results)
    return results_df

In [11]:
# Nueva funcionalidad procesamiento de datos clinicos
def process_cbioportal_metadata(clinical_path, sample_path, clinical_attribute):
    """
    Procesar metadatos de cBioPortal - versión Python de la función R.

    Args:
        clinical_path: Ruta al archivo de datos clínicos
        sample_path: Ruta al archivo de datos de muestra
        clinical_attribute: Atributo clínico a procesar

    Returns:
        dict: Diccionario con metadata procesada y resumen
    """
    print("Cargando dataset de metadatos de cBioPortal...")

    # Cargar archivos (skip=4 para saltar las primeras 4 líneas)
    clinical_data = pd.read_csv(clinical_path, sep='\t', skiprows=4)
    sample_data = pd.read_csv(sample_path, sep='\t', skiprows=4)

    ##### Procesar datos clínicos #####
    # Filtrar solo las columnas PATIENT_ID y el atributo clínico
    clinical_data = clinical_data[['PATIENT_ID', clinical_attribute]]

    # Eliminar filas con NA en el atributo clínico
    clinical_data = clinical_data.dropna(subset=[clinical_attribute])

    # Si la columna es de tipo texto, limpiar y convertir a mayúsculas
    if clinical_data[clinical_attribute].dtype == 'object':
        # Eliminar espacios al inicio y al final, luego convertir a mayúsculas
        clinical_data[clinical_attribute] = clinical_data[clinical_attribute].astype(str).str.strip().str.upper()

    # Eliminar filas duplicadas
    clinical_data = clinical_data.drop_duplicates()

    ##### Procesar datos de muestra #####
    sample_data = sample_data[['SAMPLE_ID', 'PATIENT_ID']]

    ##### Fusionar datos clínicos y de muestra #####
    metadata = pd.merge(sample_data, clinical_data, on='PATIENT_ID', how='inner')
    metadata = metadata[['SAMPLE_ID', clinical_attribute]]

    # Reemplazar guiones con puntos en SAMPLE_ID
    metadata['SAMPLE_ID'] = metadata['SAMPLE_ID'].str.replace('-', '.')

    # Contar duplicados
    n_before = len(metadata)
    metadata = metadata.drop_duplicates()
    n_after = len(metadata)

    # Mensajes de resumen
    print("Dataset de metadatos: Resumen")
    print(f"Dataset de metadatos: Filas (muestras) eliminadas por duplicación: {n_before - n_after}")
    print(f"Dataset de metadatos: Número de muestras: {n_after}")

    # Resumen por atributo clínico - CORRECCIÓN AQUÍ
    summary = (metadata[clinical_attribute]
               .value_counts()
               .reset_index())
    # Corregir los nombres de las columnas después de value_counts()
    summary.columns = [clinical_attribute, 'n']
    # Asegurarse de que 'n' sea numérico
    summary['n'] = pd.to_numeric(summary['n'], errors='coerce')
    summary['Proportion'] = 100 * summary['n'] / summary['n'].sum()

    return {
        'metadata': metadata,
        'summary': summary
    }



In [18]:
clinical_path = "datasets/acc_tcga/data_clinical_patient.txt"
sample_path = "datasets/acc_tcga/data_clinical_sample.txt"
clinical_attribute = "SEX"

process = process_cbioportal_metadata(
    clinical_path,
    sample_path,
    clinical_attribute
)
print("\nvamos a ver que sale......")
print("\nMetadatos procesados:")
print(process['metadata'].head())
print("\nResumen de metadatos:")
print(process['summary'])


Cargando dataset de metadatos de cBioPortal...
Dataset de metadatos: Resumen
Dataset de metadatos: Filas (muestras) eliminadas por duplicación: 0
Dataset de metadatos: Número de muestras: 92

vamos a ver que sale......

Metadatos procesados:
         SAMPLE_ID     SEX
0  TCGA.OR.A5J1.01    MALE
1  TCGA.OR.A5J2.01  FEMALE
2  TCGA.OR.A5J3.01  FEMALE
3  TCGA.OR.A5J4.01  FEMALE
4  TCGA.OR.A5J5.01    MALE

Resumen de metadatos:
      SEX   n  Proportion
0  FEMALE  60   65.217391
1    MALE  32   34.782609


In [5]:
# Cargar datos
df_RNAseq = pd.read_csv("examples/RNAseq_log_TMP_acc_tcga.csv")
df_clinical = pd.read_csv("examples/clinical_data_acc_tcga.csv")

# Usar SEX como parámetro clinical_attribute
clinical_attribute = "SEX"

print("Datos cargados:")
print(f"RNA-seq shape: {df_RNAseq.shape}")
print(f"Clinical data shape: {df_clinical.shape}")
print(f"Unique values in {clinical_attribute}: {df_clinical[clinical_attribute].unique()}")


Datos cargados:
RNA-seq shape: (17436, 80)
Clinical data shape: (79, 3)
Unique values in SEX: ['MALE' 'FEMALE']


In [6]:
# Preparar datos para el análisis
# Asegurarse de que los datos de RNA-seq estén en el formato correcto
# (genes en filas, muestras en columnas)

# Si la primera columna contiene nombres de genes, usar como índice
if 'Unnamed: 0' in df_RNAseq.columns:
    df_RNAseq = df_RNAseq.set_index('Unnamed: 0')

sample_column = 'SAMPLE_ID'
if sample_column in df_clinical.columns:
    common_samples = sorted(set(df_RNAseq.columns) & set(df_clinical[sample_column]))
    df_RNAseq_filtered = df_RNAseq[common_samples]
    df_clinical_filtered = df_clinical[df_clinical[sample_column].isin(common_samples)]
    df_clinical_filtered = df_clinical_filtered.set_index(sample_column).reindex(common_samples).reset_index()
else:
    df_RNAseq_filtered = df_RNAseq
    df_clinical_filtered = df_clinical

print(f"Datos filtrados - RNA-seq: {df_RNAseq_filtered.shape}, Clinical: {df_clinical_filtered.shape}")


Datos filtrados - RNA-seq: (17436, 79), Clinical: (79, 3)


In [7]:
# Ejecutar análisis de expresión diferencial
try:
    results = perform_differential_expression(
        df_RNAseq_filtered, 
        df_clinical_filtered, 
        clinical_attribute
    )
    
    print("Análisis completado exitosamente!")
    print(f"Resultados shape: {results.shape}")
    print("\nPrimeras filas de los resultados:")
    print(results.head())
    
    # Mostrar genes más significativos
    # print("\nGenes más significativos (top 10):")
    top_genes = results.nsmallest(10, 'adj.P.Val')
    # print(top_genes[['logFC', 'P.Value', 'adj.P.Val']])

    top_genes_reset = top_genes.reset_index()  # El índice (nombre del gen) pasa a columna
    # API response
    response = {
        "status": "success",
        "message": "Differential expression analysis completed successfully.",
        "results": top_genes_reset.to_dict(orient='records')
    }
    print("\nResponse JSON:" + json.dumps(response, indent=2))
    
except Exception as e:
    print(f"Error en el análisis: {e}")
    print("Verificar que R y el paquete limma estén instalados correctamente")


Análisis completado exitosamente!
Resultados shape: (17436, 6)

Primeras filas de los resultados:
            logFC   AveExpr          t       P.Value     adj.P.Val          B
RPS4Y1  10.034285  4.653869  26.283862  5.342039e-41  9.314380e-37  71.958680
DDX3Y    8.710125  3.798471  25.183240  1.122321e-39  9.784395e-36  69.825891
HY       8.109795  3.482668  21.565935  5.415642e-35  3.147571e-31  61.889254
PRKY     4.756050  2.243615  21.470226  7.331222e-35  3.195680e-31  61.658213
XIST    -9.489872  7.844222 -19.925844  1.112824e-32  3.880641e-29  57.766025

Response JSON:{
  "status": "success",
  "message": "Differential expression analysis completed successfully.",
  "results": [
    {
      "index": "RPS4Y1",
      "logFC": 10.034285180416521,
      "AveExpr": 4.653869164920898,
      "t": 26.28386175615271,
      "P.Value": 5.342039386486151e-41,
      "adj.P.Val": 9.314379874277253e-37,
      "B": 71.95867969269098
    },
    {
      "index": "DDX3Y",
      "logFC": 8.710124932

In [8]:
# Formatea todas las columnas numéricas a notación científica con 2 decimales
def format_scientific(df, decimals=4):
    df_fmt = df.copy()
    for col in df_fmt.select_dtypes(include=[np.number]).columns:
        df_fmt[col] = df_fmt[col].apply(lambda x: f"{x:.{decimals}e}")
    return df_fmt

results_fmt = format_scientific(results)

# Exporta a HTML con los números en notación científica
html = results_fmt.to_html(classes='table table-striped', index=True)
with open('examples/informe_resultados_completo.html', 'w') as f:
    f.write("""
    <html>
    <head>
      <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.7/css/bootstrap.min.css">
    </head>
    <body>
    <div class="container">
    <h2>Informe de Resultados</h2>
    """)
    f.write(html)
    f.write("""
    </div>
    </body>
    </html>
    """)



In [8]:
# Función auxiliar para verificar instalación de R y limma
def check_r_installation():
    """Verificar que R y limma estén disponibles"""
    try:
        # Verificar R
        r_version = robjects.r('R.version.string')[0]
        print(f"R version: {r_version}")
        
        # Verificar limma
        limma = importr('limma')
        print("Limma package loaded successfully")
        
        return True
    except Exception as e:
        print(f"Error checking R installation: {e}")
        print("Asegúrate de tener R instalado y el paquete limma disponible")
        print("Para instalar limma en R: install.packages('BiocManager'); BiocManager::install('limma')")
        return False

# Verificar instalación
check_r_installation()

R version: R version 4.5.0 (2025-04-11)
Limma package loaded successfully


True